# Yelping For A Second Helping!

#### Brookie: Sunidhi Ayyagari, Sarah Ding, Sophie Gill, Stephanie Ou Yang

### Data Cleaning and Preprocessing

Download the dataset via kagglehub

Dataset can be found here: https://www.kaggle.com/datasets/omkarsabnis/yelp-reviews-dataset/data

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("omkarsabnis/yelp-reviews-dataset")

print("Path to dataset files:", path)
print("Files:", os.listdir(path))

Path to dataset files: /kaggle/input/yelp-reviews-dataset
Files: ['yelp.csv']


Read the dataset from the folder into a DataFrame.

In [ ]:
import pandas as pd

file_path = os.path.join(path, 'yelp.csv')
yelp_dataset = pd.read_csv(file_path)

In [ ]:
yelp_dataset.dropna(inplace=True)
yelp_dataset.drop_duplicates(inplace=True)
print(len(yelp_dataset))

10000


In [ ]:
import re

# Lowercase reivews
yelp_dataset['cleaned_text'] = yelp_dataset['text'].str.lower()
# Remove non-alphabetical characters and puntuation
yelp_dataset['cleaned_text'] = yelp_dataset['cleaned_text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords

# Remove stopwords
stop_words = set(stopwords.words('english'))
yelp_dataset['cleaned_text'] = yelp_dataset['cleaned_text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

# Lemmatize
yelp_dataset['cleaned_text'] = yelp_dataset['cleaned_text'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

In [ ]:
print('Yelp review before text cleaning and preprocessing:')
print(yelp_dataset['text'][0])

print('\nYelp review after text cleaning and preprocessing:')
print(yelp_dataset['cleaned_text'][0])

Yelp review before text cleaning and preprocessing:
My wife took me here on my birthday for breakfast and it was excellent.  The weather was perfect which made sitting outside overlooking their grounds an absolute pleasure.  Our waitress was excellent and our food arrived quickly on the semi-busy Saturday morning.  It looked like the place fills up pretty quickly so the earlier you get here the better.

Do yourself a favor and get their Bloody Mary.  It was phenomenal and simply the best I've ever had.  I'm pretty sure they only use ingredients from their garden and blend them fresh when you order it.  It was amazing.

While EVERYTHING on the menu looks excellent, I had the white truffle scrambled eggs vegetable skillet and it was tasty and delicious.  It came with 2 pieces of their griddled bread with was amazing and it absolutely made the meal complete.  It was the best "toast" I've ever had.

Anyway, I can't wait to go back!

Yelp review after text cleaning and preprocessing:
wife t

Map the star ratings to 3-class sentiments with

* 1-2 stars : -1
* 3 stars : 0
* 4-5 stars : 1

This will simplify the number of categories for training and label reviews with clearer sentiment scores with -1 being negative, 0 being neutral, and 1 being positive.

In [ ]:
# Map ratings to sentiment
rating_to_sentiment = {1: -1, 2: -1, 3: 0, 4: 1, 5: 1}

yelp_dataset['sentiment'] = yelp_dataset['stars'].map(rating_to_sentiment)

In [ ]:
print(yelp_dataset.columns)

Index(['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id',
       'cool', 'useful', 'funny', 'cleaned_text', 'sentiment'],
      dtype='object')


In [ ]:
yelp_dataset.drop(columns=['business_id', 'date', 'text','review_id', 'stars', 'type', 'user_id', 'cool', 'useful', 'funny'], inplace=True)
print(yelp_dataset.columns)

Index(['cleaned_text', 'sentiment'], dtype='object')


### NER

First step is to tokenize. Using a food specific tokenizer from Hugging Face that can be found here: https://huggingface.co/Dizex/InstaFoodRoBERTa-NER

> InstaFoodRoBERTa-NER is a fine-tuned BERT model that is ready to use for Named Entity Recognition of Food entities on social media like informal text (e.g. Instagram, X, Reddit). It has been trained to recognize a single entity: food (FOOD).



In [ ]:
!pip install transformers
!pip install torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

# Load the Hugging Face model
model_name = "Dizex/InstaFoodRoBERTa-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cuda:0


In [ ]:
def extract_dishes(text):
    results = ner_pipeline(text)
    return [ent['word'] for ent in results if ent['entity_group'] == 'FOOD']

yelp_dataset['dishes'] = yelp_dataset['cleaned_text'].apply(extract_dishes)
yelp_dataset.to_csv("yelp_with_dishes.csv", index=False)

In [ ]:
print(yelp_dataset['cleaned_text'][0])
print(yelp_dataset['dishes'][0])

wife took birthday breakfast excellent weather perfect made sitting outside overlooking ground absolute pleasure waitress excellent food arrived quickly semibusy saturday morning looked like place fill pretty quickly earlier get better favor get bloody mary phenomenal simply best ive ever im pretty sure use ingredient garden blend fresh order amazing everything menu look excellent white truffle scrambled egg vegetable skillet tasty delicious came piece griddled bread amazing absolutely made meal complete best toast ive ever anyway cant wait go back
[' white truffle scrambled egg vegetable skillet', ' bread', ' toast']


### Sentiment Analysis

In [ ]:
!apt update && apt install cuda-11-8
!pip install deeplabcut[tf]

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,628 B]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,150 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [4,266 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Pa

In [ ]:
!pip install spacy
!spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 116.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pip install setfit datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from setfit import AbsaModel

model = AbsaModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-MiniLM-L6-v2",
    spacy_model="en_core_web_sm"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Load dataset
yelp_dataset = pd.read_csv("yelp_with_dishes.csv")

# Prepare dataset
yelp_dataset.rename(columns={'cleaned_text': 'text', 'dishes': 'span', 'sentiment': 'label'}, inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_test_df = train_test_split(yelp_dataset, test_size=0.4)
eval_df, test_df = train_test_split(temp_test_df, test_size=0.2)

In [ ]:
# Down-sample the majority class in the training set
train_resampled_df = train_df.groupby('label').apply(lambda x: x.sample(train_df['label'].value_counts().min())).reset_index(drop=True)

<ipython-input-3-ed91caf465ea>:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_resampled_df = train_df.groupby('label').apply(lambda x: x.sample(train_df['label'].value_counts().min())).reset_index(drop=True)


In [ ]:
# Convert the resampled train and eval datasets into `Dataset` format
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_resampled_df)
eval_dataset = Dataset.from_pandas(eval_df)

train_dataset = train_dataset.add_column("ordinal", [0] * len(train_dataset))
eval_dataset = eval_dataset.add_column("ordinal", [0] * len(eval_dataset))

# Set the format for PyTorch
train_dataset.set_format("torch")
eval_dataset.set_format("torch")

In [ ]:
import random

# Set random seed for reproducibility
random.seed(42)

# Shuffle dataset and select first 1000 rows
small_train_df = train_dataset.shuffle(seed=42).select(range(500))

# Shuffle and select first 500 rows
small_eval_df = eval_dataset.shuffle(seed=42).select(range(250))

In [ ]:
import psutil
print(f"Memory Usage: {psutil.virtual_memory().percent}%")

Memory Usage: 16.2%


In [ ]:
import gc
gc.collect()

132

In [ ]:
CUDA_LAUNCH_BLOCKING=1

In [ ]:
!pip install torch

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
from setfit import AbsaTrainer, TrainingArguments, AbsaModel
from transformers import EarlyStoppingCallback, AutoTokenizer

# Re-initialize the model and tokenizer here
model = AbsaModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    spacy_model="en_core_web_sm",
)

# Load the Hugging Face tokenizer
model_name = "Dizex/InstaFoodRoBERTa-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)

args = TrainingArguments(
    output_dir="models",
    num_epochs=2,
    use_amp=True,
    batch_size=16,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    load_best_model_at_end=True,
)

trainer = AbsaTrainer(
    model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
trainer.train()

model.save_pretrained('fine_tuned_bert_model')
tokenizer.save_pretrained('fine_tuned_bert_model')

Streaming output truncated to the last 5000 lines.
The ordinal of 0 for span "[' o', 'ahu', ' ar', 'rib', 'as', ' chim', 'ich', 'anga', ' bean', ' rice', ' u', 'k', ' meat root vegetable er', 'in fajita salad', ' chicken', ' beans', 'rice']" in 'saddens say isnt worst mexican restaurant weve eaten thats oahu near waikiki beach close damn drive arribas get place chimichanga little anemic though filling ok bean tasteless rice looked part spicy average uk meal involving boiled meat root vegetable erin fajita salad similarly lack luster appearance tasty yet bland chicken usually unless there attempt chefing opposed cooking ill order basic dish dont shoot chef though trust guy safe agent sand safe first time beansrice suck return overall feel like alien space one running place learned cooking picture certainly mistake wont repeating' is too high. Skipping this sample.
The ordinal of 0 for span "[' pastry', ' espresso', ' masc', 'arp', 'one fig jam marble rye lox', ' cream cheese tomato', ' 

Map:   0%|          | 0/57766 [00:00<?, ? examples/s]

In [ ]:
X_test = test_df.drop(columns={'label'})
y_test = test_df['label']
predictions = model.predict(X_test)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Predicted labels
y_pred = np.argmax(predictions.predictions, axis=1)

# True labels
y_true = predictions.label_ids

# Accuracy
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")  # or "macro" for class-balanced

print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=["negative", "neutral", "positive"], cmap="Blues"
)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Number of classes
n_classes = 3
class_names = ["negative", "neutral", "positive"]

# Binarize true labels for ROC
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])

# Probabilities from the model
y_score = predictions.predictions

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot the ROC curves
plt.figure(figsize=(8, 6))
colors = ['red', 'orange', 'green']
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=2,
             label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest)')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def recommend_best_dish(text, dishes, model):
    from datasets import Dataset
    import numpy as np

    input_data = {
        "text": [text] * len(dishes),
        "span": dishes,
        "ordinal": [0] * len(dishes),
    }

    predict_dataset = Dataset.from_dict(input_data)
    predict_dataset.set_format("torch")

    preds = model.predict(predict_dataset)
    pred_labels = np.argmax(preds.predictions, axis=1)
    sentiment_map = {0: "negative", 1: "neutral", 2: "positive"}
    dish_sentiments = [(dish, sentiment_map[label]) for dish, label in zip(dishes, pred_labels)]

    print("Dish Sentiments:")
    for dish, sentiment in dish_sentiments:
        print(f"- {dish}: {sentiment}")

    positive_dishes = [dish for dish, sentiment in dish_sentiments if sentiment == "positive"]
    if positive_dishes:
        return f"Recommended Dish: {positive_dishes[0]}"
    else:
        return "No strongly positive dish mentioned."

In [ ]:
recommend_best_dish(
    "The truffle pasta was amazing but the sushi was disappointing.",
    ["truffle pasta", "sushi"],
    model
)